# Recommendation System Using Neural Networks
## Deep Learning Subject Project — Kartik Sehrawat

---

### What is Neural Collaborative Filtering (NCF)?

Traditional recommendation systems use **Matrix Factorization (MF)**, which decomposes the user-item interaction matrix into latent factors using a simple dot-product.  
**NCF** replaces this dot-product with a **neural network**, allowing the model to capture **non-linear, complex user-item interactions** that MF cannot model.

#### Architecture:
```
User ID → [Embedding] ─┬─ GMF branch: element-wise × ─┐
                       │                               CONCAT → Dense(1) → ŷ
Item ID → [Embedding] ─┴─ MLP branch: 64→32→16→8 ──┘
```

**Paper:** He, X. et al. (2017). *Neural Collaborative Filtering*. WWW 2017.

## Step 1: Install Required Libraries

In [ ]:
# Install dependencies (only needed once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'tensorflow', 'pandas', 'numpy', 'scikit-learn',
    'matplotlib', 'seaborn', 'requests', 'tqdm'], stdout=subprocess.DEVNULL)
print('✅ Libraries ready!')

## Step 2: Import Libraries

In [ ]:
import os, pickle, requests, zipfile, io, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')
print(f'GPU Available:      {len(tf.config.list_physical_devices("GPU")) > 0}')

## Step 3: Load the MovieLens 100K Dataset

The **MovieLens 100K** dataset contains:
- **943 users**, **1,682 movies**
- **100,000 ratings** (scale 1–5)
- Collected by GroupLens Research at the University of Minnesota

We treat this as an **implicit feedback** problem: any rating ≥ 1 is treated as a positive interaction.

In [ ]:
def load_movielens():
    data_dir = 'ml-100k'
    if not os.path.exists(data_dir):
        print('⬇ Downloading MovieLens 100K...')
        r = requests.get('https://files.grouplens.org/datasets/movielens/ml-100k.zip', timeout=60)
        zipfile.ZipFile(io.BytesIO(r.content)).extractall('.')
        print('✅ Downloaded!')

    df = pd.read_csv(
        os.path.join(data_dir, 'u.data'),
        sep='\t', names=['user_id','item_id','rating','timestamp']
    )
    movies = pd.read_csv(
        os.path.join(data_dir, 'u.item'),
        sep='|', encoding='latin-1', usecols=[0,1], names=['item_id','title']
    )
    return df, movies

df, movies_df = load_movielens()
print(f'Ratings shape: {df.shape}')
df.head()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('MovieLens 100K — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Rating distribution
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#5b8dee', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings per user (top 20)
user_counts = df.groupby('user_id').size().sort_values(ascending=False)
user_counts.head(20).plot(kind='bar', ax=axes[1], color='#a78bfa', edgecolor='white')
axes[1].set_title('Top 20 Most Active Users')
axes[1].set_xlabel('User ID')
axes[1].set_ylabel('# Ratings')
axes[1].tick_params(axis='x', rotation=90)

# Ratings per movie (top 20)
item_counts = df.groupby('item_id').size().sort_values(ascending=False)
item_counts.head(20).plot(kind='bar', ax=axes[2], color='#34d399', edgecolor='white')
axes[2].set_title('Top 20 Most Rated Movies')
axes[2].set_xlabel('Item ID')
axes[2].set_ylabel('# Ratings')
axes[2].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Sparsity: {100 * (1 - len(df) / (df.user_id.nunique() * df.item_id.nunique())):.2f}%')

## Step 5: Preprocessing — Encode User/Item IDs

In [ ]:
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: i for i, it in enumerate(item_ids)}

df['user']  = df['user_id'].map(user2idx)
df['item']  = df['item_id'].map(item2idx)
df['label'] = 1  # all observed interactions are positive

n_users = len(user2idx)
n_items = len(item2idx)
user_item_set = set(zip(df['user'], df['item']))

print(f'Users: {n_users} | Items: {n_items} | Interactions: {len(df)}')
df[['user_id','item_id','user','item','rating','label']].head()

## Step 6: Negative Sampling

Since we use **implicit feedback**, we only have positive interactions. For training, we need **negative examples** — items the user has NOT interacted with.

For each positive (user, item) pair, we sample **4 random negative items**.

In [ ]:
def negative_sample(df, n_users, n_items, num_neg=4):
    user_item_set_local = set(zip(df['user'], df['item']))
    users, items, labels = [], [], []
    for u, i in zip(df['user'], df['item']):
        users.append(u); items.append(i); labels.append(1)
        for _ in range(num_neg):
            neg = np.random.randint(0, n_items)
            while (u, neg) in user_item_set_local:
                neg = np.random.randint(0, n_items)
            users.append(u); items.append(neg); labels.append(0)
    sample = pd.DataFrame({'user': users, 'item': items, 'label': labels})
    return sample.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['label'])
train_sample = negative_sample(train_df, n_users, n_items, num_neg=4)

print(f'Training samples (with negatives): {len(train_sample)}')
print(f'Label distribution:')
print(train_sample['label'].value_counts())

## Step 7: Build the NCF Model

### Architecture Details:

| Component | Description |
|-----------|-------------|
| GMF Branch | User & Item Embeddings (dim=32) → Element-wise multiply |
| MLP Branch | User & Item Embeddings (dim=32) → Concatenate → Dense(64→32→16→8) |
| NeuMF | Concatenate GMF + MLP → Dense(1, sigmoid) |
| Loss | Binary Cross-Entropy |
| Optimizer | Adam (lr=0.001) |

In [ ]:
EMBEDDING_DIM = 32
MLP_LAYERS    = [64, 32, 16, 8]

def build_ncf(n_users, n_items, embedding_dim=32, mlp_layers=None):
    if mlp_layers is None:
        mlp_layers = [64, 32, 16, 8]

    user_input = layers.Input(shape=(1,), name='user_input')
    item_input = layers.Input(shape=(1,), name='item_input')

    # GMF Branch
    gmf_user = layers.Flatten()(layers.Embedding(n_users, embedding_dim, name='gmf_user_emb')(user_input))
    gmf_item = layers.Flatten()(layers.Embedding(n_items, embedding_dim, name='gmf_item_emb')(item_input))
    gmf_out  = layers.Multiply(name='gmf_multiply')([gmf_user, gmf_item])

    # MLP Branch
    mlp_user = layers.Flatten()(layers.Embedding(n_users, mlp_layers[0]//2, name='mlp_user_emb')(user_input))
    mlp_item = layers.Flatten()(layers.Embedding(n_items, mlp_layers[0]//2, name='mlp_item_emb')(item_input))
    x = layers.Concatenate()([mlp_user, mlp_item])
    for i, units in enumerate(mlp_layers):
        x = layers.Dense(units, activation='relu', name=f'mlp_dense_{i}')(x)
        x = layers.Dropout(0.2)(x)

    # NeuMF Fusion
    combined = layers.Concatenate(name='neumf_concat')([gmf_out, x])
    output   = layers.Dense(1, activation='sigmoid', name='output')(combined)

    model = Model(inputs=[user_input, item_input], outputs=output, name='NeuralCF')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_ncf(n_users, n_items, EMBEDDING_DIM, MLP_LAYERS)
model.summary()

## Step 8: Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    [train_sample['user'].values, train_sample['item'].values],
    train_sample['label'].values,
    batch_size=256,
    epochs=20,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## Step 9: Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NCF Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'],     label='Train Loss', color='#4C9BE8', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#E8834C', linewidth=2, linestyle='--')
axes[0].set_title('Binary Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc', color='#4CE8A0', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='#E84C6E', linewidth=2, linestyle='--')
axes[1].set_title('Classification Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('training_curves.png', dpi=120, bbox_inches='tight'); plt.show()

## Step 10: Evaluate — Hit Rate@10 & NDCG@10

**Evaluation Protocol (Leave-One-Out):**
1. For each user, hold out their last interaction as the "ground truth" positive
2. Sample 99 random items the user hasn't seen as negatives
3. Score all 100 items with the model
4. Check if the positive item appears in the Top 10

**Metrics:**
- **Hit Rate@10 (HR@10):** What fraction of users have the positive item in their top-10 predictions
- **NDCG@10:** Accounts for the rank of the positive item — higher rank = higher score

In [ ]:
TOP_K = 10
hits, ndcgs = [], []

for user_idx, group in tqdm(test_df.groupby('user'), desc='Evaluating'):
    pos_items = group[group['label'] == 1]['item'].tolist()
    if not pos_items: continue
    pos_item = pos_items[0]
    negatives = []
    while len(negatives) < 99:
        neg = np.random.randint(0, n_items)
        if neg != pos_item and (user_idx, neg) not in user_item_set:
            negatives.append(neg)
    eval_items = [pos_item] + negatives
    preds = model.predict(
        [np.array([user_idx]*100), np.array(eval_items)],
        batch_size=100, verbose=0
    ).flatten()
    top_k_indices = np.argsort(preds)[::-1][:TOP_K]
    top_k_items   = [eval_items[i] for i in top_k_indices]
    hit = 1 if pos_item in top_k_items else 0
    hits.append(hit)
    ndcgs.append(1.0/np.log2(top_k_items.index(pos_item)+2) if hit else 0.0)

hr   = np.mean(hits)
ndcg = np.mean(ndcgs)
print(f'\n📊 Evaluation Results:')
print(f'  Hit Rate@{TOP_K}  = {hr:.4f}  ({hr*100:.1f}%)')
print(f'  NDCG@{TOP_K}      = {ndcg:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar([f'Hit Rate@{TOP_K}', f'NDCG@{TOP_K}'], [hr, ndcg],
              color=['#4C9BE8', '#E8834C'], width=0.4, edgecolor='white')
ax.set_ylim(0, 1)
ax.set_title('Recommendation Quality Metrics', fontsize=13, fontweight='bold')
for bar, val in zip(bars, [hr, ndcg]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=120, bbox_inches='tight'); plt.show()

## Step 11: Generate Recommendations for a Specific User

In [ ]:
def recommend_for_user(user_id_original, n=10):
    if user_id_original not in user2idx:
        print(f'User {user_id_original} not found!')
        return
    user_idx = user2idx[user_id_original]
    idx2item = {v: k for k, v in item2idx.items()}

    seen = {i for (u, i) in user_item_set if u == user_idx}
    candidates = [i for i in range(n_items) if i not in seen]
    preds = model.predict(
        [np.array([user_idx]*len(candidates)), np.array(candidates)],
        batch_size=512, verbose=0
    ).flatten()

    top_indices = np.argsort(preds)[::-1][:n]
    print(f'\n🎬 Top {n} Recommendations for User #{user_id_original}:')
    print(f'{"Rank":<6} {"Movie Title":<55} {"Score":<8}')
    print('-'*70)
    for rank, idx in enumerate(top_indices, 1):
        item_idx = candidates[idx]
        orig_id  = idx2item[item_idx]
        row = movies_df[movies_df['item_id'] == orig_id]
        title = row['title'].values[0] if len(row) > 0 else f'Movie #{orig_id}'
        score = preds[idx]
        print(f'{rank:<6} {title:<55} {score:.4f}')

# Try different user IDs!
recommend_for_user(1,  n=10)
recommend_for_user(25, n=10)
recommend_for_user(50, n=10)

## Step 12: Save the Model

In [ ]:
model.save('model.h5')
with open('encodings.pkl', 'wb') as f:
    pickle.dump({
        'user2idx': user2idx, 'item2idx': item2idx,
        'n_users': n_users, 'n_items': n_items,
        'user_item_set': user_item_set, 'hr': hr, 'ndcg': ndcg, 'top_k': TOP_K
    }, f)
print('✅ model.h5 and encodings.pkl saved!')
print('\n▶ Now run: python app.py   →   open http://127.0.0.1:5000')

---

## Summary

| Step | Description |
|------|-------------|
| 1 | Loaded MovieLens 100K (943 users, 1682 movies, 100K ratings) |
| 2 | Preprocessed: encoded IDs, treated as implicit feedback |
| 3 | Applied Negative Sampling (4 negatives per positive) |
| 4 | Built NCF Model (GMF + MLP branches, ~300K parameters) |
| 5 | Trained with Binary Cross-Entropy, Adam optimizer |
| 6 | Evaluated using Hit Rate@10 and NDCG@10 |
| 7 | Generated personalized Top-10 movie recommendations |

**Reference:** He, X., Liao, L., Zhang, H., Nie, L., Hu, X., & Chua, T.S. (2017). *Neural collaborative filtering*. In Proceedings of the 26th international conference on world wide web (pp. 173-182).